In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_submission=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [4]:
# import os
# import random
# import torch
# import wandb

# from datasets import Dataset
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, f1_score

# from transformers import (
#     AutoTokenizer,
#     AutoModelForMultipleChoice,
#     TrainingArguments,
#     Trainer,
# )

In [5]:
train.shape,test.shape

((2000, 8), (500, 7))

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [8]:
train.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [9]:
# MODEL="microsoft/deberta-v3-large"
# MAX_LEN=384

In [10]:
# label2id={"A":0,"B":1,"C":2,"D":3,"E":4}
# id2label={v:k for k,v in label2id.items()}
# train["label"]=train.answer.map(label2id)

In [11]:
# def build(df):
#     txt=[]
#     for _,r in df.iterrows():
#         txt.append(
# f"""Question:
# {r['prompt']}

# A. {r['A']}
# B. {r['B']}
# C. {r['C']}
# D. {r['D']}
# E. {r['E']}"""
# )
#     return txt

# train_ds=Dataset.from_dict({"text":build(train),"label":train.label.tolist()})
# test_ds=Dataset.from_dict({"text":build(test)})

# tok=AutoTokenizer.from_pretrained(MODEL)

In [12]:
# def tokenize(x):
#     return tok(x["text"],truncation=True,max_length=MAX_LEN)

# train_ds=train_ds.map(tokenize,batched=True)
# test_ds=test_ds.map(tokenize,batched=True)

# model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=5)

# args = TrainingArguments(
#     output_dir="model",
#     learning_rate=2e-5,
#     per_device_train_batch_size=8,
#     num_train_epochs=3,
#     fp16=torch.cuda.is_available(),
#     eval_strategy="no",
#     save_strategy="epoch",
#     report_to="none"
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_ds,
#     processing_class=tok
# )


In [13]:
# pred=trainer.predict(test_ds).predictions
# prob=torch.softmax(torch.tensor(pred),dim=1).numpy()

# labels=np.array(["A","B","C","D","E"])
# top3=[" ".join(labels[np.argsort(-p)[:3]]) for p in prob]



In [14]:
# sub=pd.DataFrame({"id":test.id,"Prediction":top3})
# sub.to_csv("submission.csv",index=False)
# print(sub.head())

===================================================================

In [15]:
# ==========================================
# Imports
# ==========================================

import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

In [16]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

wandb.login(
    key=user_secrets.get_secret("WANDB_API_KEY")
)

wandb.init(
    project="mcq-challenge",
    name="bge-large-logreg"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2006126 (24f2006126-iitm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run j448uc45
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260627_102206-j448uc45
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bge-large-logreg
wandb: ⭐️ View project at https://wandb.ai/24f2006126-iitm/mcq-challenge
wandb: 🚀 View run at https://wandb.ai/24f2006126-iitm/mcq-challenge/runs/j448uc45


In [17]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [18]:
# ==========================================
# Answer Options
# ==========================================

OPTIONS = ["A", "B", "C", "D", "E"]

In [19]:
# ==========================================
# Build Binary Training Dataset
# ==========================================

def build_train(df):

    texts = []
    labels = []
    qids = []
    options = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        prompt = str(row["prompt"])

        answer = row["answer"]

        for op in OPTIONS:

            candidate = str(row[op])

            text = f"""
Question:
{prompt}

Candidate Answer:
{candidate}
"""

            texts.append(text)

            labels.append(int(op == answer))

            qids.append(row["id"])

            options.append(op)

    return texts, np.array(labels), qids, options

In [20]:
train_texts, train_labels, train_qids, train_options = build_train(train)

print("Training samples:", len(train_texts))
print("Positive labels :", train_labels.sum())
print("Negative labels :", len(train_labels) - train_labels.sum())

  0%|          | 0/2000 [00:00<?, ?it/s]

Training samples: 10000
Positive labels : 2000
Negative labels : 8000


In [21]:
# ==========================================
# Build Test Dataset
# ==========================================

def build_test(df):

    texts = []
    qids = []
    options = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        prompt = str(row["prompt"])

        for op in OPTIONS:

            candidate = str(row[op])

            text = f"""
Question:
{prompt}

Candidate Answer:
{candidate}
"""

            texts.append(text)

            qids.append(row["id"])

            options.append(op)

    return texts, qids, options

In [22]:
test_texts, test_qids, test_options = build_test(test)

print("Test samples:", len(test_texts))

  0%|          | 0/500 [00:00<?, ?it/s]

Test samples: 2500


In [23]:
print(pd.Series(train_labels).value_counts())

0    8000
1    2000
Name: count, dtype: int64


In [24]:
sample_idx = 0

print("Question ID :", train_qids[sample_idx])
print("Option      :", train_options[sample_idx])
print("Label       :", train_labels[sample_idx])
print()
print(train_texts[sample_idx])

Question ID : 1
Option      : A
Label       : 0


Question:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Candidate Answer:
Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.



In [25]:
# ==========================================
# Sentence Embeddings
# ==========================================

from sentence_transformers import SentenceTransformer
import torch

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [27]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

embedder = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded.


In [28]:
X_train = embedder.encode(
    train_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Train embeddings shape:", X_train.shape)

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Train embeddings shape: (10000, 1024)


In [29]:
X_test = embedder.encode(
    test_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Test embeddings shape:", X_test.shape)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Test embeddings shape: (2500, 1024)


In [30]:
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)

print("Embeddings saved.")

Embeddings saved.


In [31]:
X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")

In [32]:
import gc

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [33]:
from sklearn.model_selection import train_test_split

train_ids, valid_ids = train_test_split(
    train["id"],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_ids = set(train_ids)
valid_ids = set(valid_ids)

In [34]:
train_mask = np.array([qid in train_ids for qid in train_qids])
valid_mask = np.array([qid in valid_ids for qid in train_qids])

X_tr = X_train[train_mask]
X_val = X_train[valid_mask]

y_tr = train_labels[train_mask]
y_val = train_labels[valid_mask]

qid_val = np.array(train_qids)[valid_mask]
option_val = np.array(train_options)[valid_mask]

print(X_tr.shape)
print(X_val.shape)

(8000, 1024)
(2000, 1024)


In [35]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(

    C=4,

    max_iter=5000,

    class_weight="balanced",

    solver="lbfgs",

    random_state=42,

    n_jobs=-1

)

clf.fit(X_tr, y_tr)

LogisticRegression(C=4, class_weight='balanced', max_iter=5000, n_jobs=-1,
                   random_state=42)

In [36]:
val_prob = clf.predict_proba(X_val)[:,1]

val_pred = (val_prob >= 0.5).astype(int)

In [37]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_val, val_pred)

print("Accuracy:", acc)

Accuracy: 0.704


In [38]:
from sklearn.metrics import f1_score

f1 = f1_score(y_val, val_pred)

print("F1:", f1)

F1: 0.4896551724137931


In [39]:
# ==========================================
# Validation Predictions
# ==========================================

valid_df = pd.DataFrame({
    "id": qid_val,
    "option": option_val,
    "label": y_val,
    "score": val_prob
})

valid_df.head()

,id,option,label,score
0,24,A,0,0.362014
1,24,B,0,0.456662
2,24,C,0,0.366306
3,24,D,1,0.510967
4,24,E,0,0.411754


In [40]:
valid_df = valid_df.sort_values(
    ["id", "score"],
    ascending=[True, False]
)

valid_df.head(10)

,id,option,label,score
3,24,D,1,0.510967
1,24,B,0,0.456662
4,24,E,0,0.411754
2,24,C,0,0.366306
0,24,A,0,0.362014
6,30,B,1,0.682382
9,30,E,0,0.297420
5,30,A,0,0.295820
7,30,C,0,0.277899
8,30,D,0,0.277157


In [41]:
def map3(df):

    total = 0.0

    groups = df.groupby("id")

    for _, g in groups:

        pred = g["option"].tolist()[:3]

        gt = g.loc[g["label"] == 1, "option"].values[0]

        if pred[0] == gt:
            total += 1.0

        elif len(pred) > 1 and pred[1] == gt:
            total += 0.5

        elif len(pred) > 2 and pred[2] == gt:
            total += 1.0 / 3.0

    return total / len(groups)

In [42]:
map3_score = map3(valid_df)

print(f"MAP@3 : {map3_score:.5f}")

MAP@3 : 0.69250


In [43]:
top3 = (
    valid_df
    .groupby("id")["option"]
    .apply(lambda x: " ".join(x.head(3)))
    .reset_index()
)

top3.head()

,id,option
0,24,D B E
1,30,B E A
2,31,C D B
3,33,C B D
4,45,B E D


In [44]:
wandb.log({
    "Accuracy": acc,
    "F1": f1,
    "MAP@3": map3_score
})

In [45]:
# ==========================================
# Train Final Logistic Regression Model
# ==========================================

from sklearn.linear_model import LogisticRegression

final_model = LogisticRegression(
    C=4,
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, train_labels)

print("Training Complete!")

Training Complete!


In [46]:
# ==========================================
# Predict Test
# ==========================================

test_prob = final_model.predict_proba(X_test)[:,1]

print(test_prob.shape)

(2500,)


In [47]:
pred_df = pd.DataFrame({

    "id": test_qids,

    "option": test_options,

    "score": test_prob

})

pred_df.head()

,id,option,score
0,1,A,0.362771
1,1,B,0.411113
2,1,C,0.573434
3,1,D,0.413772
4,1,E,0.457206


In [48]:
pred_df = pred_df.sort_values(

    ["id","score"],

    ascending=[True,False]

)

In [49]:
submission = (

    pred_df.groupby("id")["option"]

    .apply(lambda x: " ".join(x.head(3)))

    .reset_index()

)

submission.columns=["id","Prediction"]

submission.head()

,id,Prediction
0,1,C E D
1,2,B D E
2,3,B E C
3,4,C B E
4,5,A C E


In [50]:
submission.to_csv(

    "submission.csv",

    index=False

)

print("submission.csv saved.")

submission.csv saved.


In [51]:
# import os 
# import re 
# import numpy as np 
# import pandas as pd 
# from collections import Counter 
# import torch 
# import torch.nn as nn 
# from torch.utils.data import Dataset, DataLoader 
# from sklearn.model_selection import train_test_split

**Dataset**

In [52]:
# train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
# sample_submission=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [53]:
# train.shape,test.shape

In [54]:
# train.head(3)

In [55]:
# test.head(2)

In [56]:
# train.info()

**Model-1**:
Attention-based BiLSTM Ranker trained from scratch using BCE loss.

**config**

In [57]:
# MAX_LEN = 256 
# EMBED_DIM = 256 
# HIDDEN_DIM = 256 
# BATCH_SIZE = 64 
# EPOCHS = 10 
# LR = 1e-3 
# LABELS = ["A", "B", "C", "D", "E"]

In [58]:
# #Tokenizer

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"[^a-z0-9 ]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# def tokenize(text):
#     return clean_text(text).split()


In [59]:
# #Vocab

# def build_vocab(df):

#     counter = Counter()

#     for _, row in df.iterrows():

#         counter.update(tokenize(row["prompt"]))

#         for col in LABELS:
#             counter.update(tokenize(row[col]))

#     vocab = {
#         "<PAD>": 0,
#         "<UNK>": 1
#     }

#     for word, freq in counter.items():
#         if freq >= 2:
#             vocab[word] = len(vocab)

#     return vocab

# def encode(text, vocab):

#     tokens = tokenize(text)

#     ids = [vocab.get(t, 1) for t in tokens]

#     ids = ids[:MAX_LEN]

#     if len(ids) < MAX_LEN:
#         ids += [0] * (MAX_LEN - len(ids))

#     return ids


In [60]:
# #Dataset

# class MCQDataset(Dataset):

#     def __init__(self, df, vocab, train_mode=True):

#         self.samples = []

#         for _, row in df.iterrows():

#             prompt = str(row["prompt"])

#             if train_mode:

#                 answer = row["answer"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     target = 1 if label == answer else 0

#                     self.samples.append(
#                         (
#                             encode(text, vocab),
#                             target
#                         )
#                     )

#             else:

#                 question_id = row["id"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     self.samples.append(
#                         (
#                             question_id,
#                             label,
#                             encode(text, vocab)
#                         )
#                     )

#         self.train_mode = train_mode

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):

#         if self.train_mode:

#             x, y = self.samples[idx]

#             return (
#                 torch.tensor(x, dtype=torch.long),
#                 torch.tensor(y, dtype=torch.float)
#             )

#         qid, label, x = self.samples[idx]

#         return (
#             qid,
#             label,
#             torch.tensor(x, dtype=torch.long)
#         )

In [61]:
# # Attention

# class AttentionPool(nn.Module):

#     def __init__(self, hidden_size):

#         super().__init__()

#         self.attn = nn.Linear(hidden_size, 1)

#     def forward(self, x):

#         scores = self.attn(x)

#         weights = torch.softmax(scores, dim=1)

#         pooled = (weights * x).sum(dim=1)

#         return pooled

In [62]:
# # Model

# class MCQRanker(nn.Module):

#     def __init__(self, vocab_size):

#         super().__init__()

#         self.embedding = nn.Embedding(
#             vocab_size,
#             EMBED_DIM,
#             padding_idx=0
#         )

#         self.lstm = nn.LSTM(
#             EMBED_DIM,
#             HIDDEN_DIM,
#             batch_first=True,
#             bidirectional=True
#         )

#         self.pool = AttentionPool(HIDDEN_DIM * 2)

#         self.classifier = nn.Sequential(
#             nn.Linear(HIDDEN_DIM * 2, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )

#     def forward(self, x):

#         emb = self.embedding(x)

#         out, _ = self.lstm(emb)

#         pooled = self.pool(out)

#         logits = self.classifier(pooled)

#         return logits.squeeze(-1)

In [63]:
# #Train...................
# def train_epoch(model, loader, optimizer, criterion):

#     model.train()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         optimizer.zero_grad()

#         logits = model(x)

#         loss = criterion(logits, y)

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     return total_loss / len(loader)

In [64]:
# #Validation
# @torch.no_grad()
# def evaluate(model, loader, criterion):

#     model.eval()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         logits = model(x)

#         loss = criterion(logits, y)

#         total_loss += loss.item()

#     return total_loss / len(loader)


In [65]:
# #Training the model................
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# vocab = build_vocab(train_df)

# train_part, valid_part = train_test_split(
#     train_df,
#     test_size=0.1,
#     random_state=42,
#     stratify=train_df["answer"]
# )

# train_ds = MCQDataset(train_part, vocab, True)
# valid_ds = MCQDataset(valid_part, vocab, True)

# train_loader = DataLoader(
#     train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )

# valid_loader = DataLoader(
#     valid_ds,
#     batch_size=BATCH_SIZE
# )

# model = MCQRanker(len(vocab)).to(DEVICE)

# criterion = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=LR
# )

# best_loss = 999

# for epoch in range(EPOCHS):

#     train_loss = train_epoch(
#         model,
#         train_loader,
#         optimizer,
#         criterion
#     )

#     valid_loss = evaluate(
#         model,
#         valid_loader,
#         criterion
#     )

#     print(
#         f"Epoch {epoch+1}/{EPOCHS} "
#         f"Train_loss={train_loss:.4f} "
#         f"Validation_loss={valid_loss:.4f}"
#     )

#     if valid_loss < best_loss:

#         best_loss = valid_loss

#         torch.save(
#             {
#                 "model": model.state_dict(),
#                 "vocab": vocab
#             },
#             "best_model.pt"
#         )

In [66]:
# #Inference
# checkpoint = torch.load(
#     "best_model.pt",
#     map_location=DEVICE
# )

# model.load_state_dict(checkpoint["model"])
# model.eval()

# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# test_ds = MCQDataset(
#     test_df,
#     vocab,
#     train_mode=False
# )

# test_loader = DataLoader(
#     test_ds,
#     batch_size=256,
#     shuffle=False
# )

# # Store scores for each question
# scores = {}

# with torch.no_grad():

#     for qids, labels, x in test_loader:

#         x = x.to(DEVICE)

#         logits = model(x)

#         probs = torch.sigmoid(logits).cpu().numpy()

#         for qid, label, p in zip(qids, labels, probs):

#             # Convert ID to string for consistency
#             qid = str(qid)

#             if qid not in scores:
#                 scores[qid] = {}

#             scores[qid][label] = float(p)
# predictions = []

# for qid in test_df["id"]:

#     qid = str(qid)

#     # Safety fallback
#     if qid not in scores:

#         predictions.append("A B C")
#         continue

#     ranked = sorted(
#         scores[qid].items(),
#         key=lambda x: x[1],
#         reverse=True
#     )

#     top3 = [label for label, score in ranked[:3]]

#     # If somehow fewer than 3 labels exist
#     while len(top3) < 3:
#         for lbl in ["A", "B", "C", "D", "E"]:
#             if lbl not in top3:
#                 top3.append(lbl)
#             if len(top3) == 3:
#                 break

#     predictions.append(" ".join(top3))

**Submission**

In [67]:
# submission = pd.DataFrame(
#     {
#         "ID": test_df["id"],
#         "Prediction": predictions
#     }
# )

# submission.to_csv(
#     "submission.csv",
#     index=False
# )

# print(submission.head())
# print("submission.csv is created successfully")

**Model-2**

In [68]:
# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torch.optim import AdamW
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from sklearn.metrics import accuracy_score, f1_score
# import wandb

In [69]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## Milestone-1 ##

**Q1**:
Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [70]:
# freq = train["answer"].value_counts().sort_index()
# print(freq)



In [71]:
# result = freq.max() + freq.min()
# print("Sum =", result)

ANS:Sum = 814

**Q2:**
After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [72]:
# import string
# vocab = set()

# for text in train["prompt"].astype(str):
#     # lowercase
#     text = text.lower()

#     # remove standard punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))  #Replace nothing,with nothing,Delete all punctuation characters

#     # split by whitespace
#     vocab.update(text.split())

# print("Vocabulary size:", len(vocab))

ANS:Vocabulary size: 859

**Q3:**
Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [73]:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# # clean
# prompt = prompt.lower()
# prompt = prompt.translate(str.maketrans('', '', string.punctuation))

# # remove stop words
# words = [word for word in prompt.split() if word not in ENGLISH_STOP_WORDS]

# print(words)
# print(len(words))

ANS:13

**Q4:**
Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer? 

In [74]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# # combine prompt + options
# texts = (
#     train["prompt"] + " " +
#     train["A"] + " " +
#     train["B"] + " " +
#     train["C"] + " " +
#     train["D"] + " " +
#     train["E"]).tolist()

# vect= TfidfVectorizer(stop_words="english")
# X = vect.fit_transform(texts)
# print("Vocabulary size:", len(vect.vocabulary_))
# print("Feature matrix shape:", X.shape)

ANS:Vocabulary size: 2762

**Q5:**
Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [75]:
# from sklearn.metrics.pairwise import cosine_similarity
# row = train[train["id"] == 1].iloc[0]

# prompt_text = row["prompt"]
# option_A_text = row["A"]

# v_prompt = vect.transform([prompt_text])
# v_A = vect.transform([option_A_text])
# score = cosine_similarity(v_prompt, v_A)[0][0]

# print(round(score, 5))

ANS:0.27202

**Q6:**
Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [76]:
# vect.fit(texts)
# options = ["A", "B", "C", "D", "E"]
# correct = 0

# for i, row in train.iterrows():
#     prompt_vec = vect.transform([row["prompt"]])

#     similarities = []

#     for opt in options:
#         opt_vec = vect.transform([row[opt]])
#         sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
#         similarities.append(sim)

#     predicted_index = np.argmax(similarities)  #argmax() finds highest value
#     predicted_answer = options[predicted_index]

#     if predicted_answer == row["answer"]:
#         correct += 1
# accuracy = correct / len(train) * 100
# print("Accuracy:", round(accuracy, 2), "%")  #This tells in what % of questions, the option most similar to the prompt was actually correct

ANS:Accuracy: 13.55 %

**Q7:**
If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  
**ANS:**
Correct answer C is at rank 1,

so,MAP@3=1/1​

        =1

**Q8:**
If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

**ANS:**
Rank of B = 2

MAP@3=1/2

=0.5

**Q9:**
The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [77]:
# freq = train["answer"].value_counts()
# top3 = freq.index[:3].tolist()

# map3 = train["answer"].apply(
#     lambda x: 1 if x==top3[0] else (0.5 if x==top3[1] else (1/3 if x==top3[2] else 0))
# ).mean()

# print(map3)

ANS:0.42125

**Q10:**
The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [78]:
# def map3_score(true, preds):  #MAP@3 function
#     if true == preds[0]:
#         return 1.0
#     elif true == preds[1]:
#         return 0.5
#     elif true == preds[2]:
#         return 1/3
#     else:
#         return 0.0

# #Evaluate all rows
# scores = []

# for _,row in train.iterrows():

#     prompt_vec = vect.transform([row["prompt"]])

#     sims = []

#     for opt in options:
#         opt_vec = vect.transform([row[opt]])
#         sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
#         sims.append((opt, sim))

#     # sort by similarity (descending)
#     ranked = sorted(sims, key=lambda x: x[1], reverse=True)

#     top3 = [r[0] for r in ranked[:3]]

#     scores.append(map3_score(row["answer"], top3))

# #Final MAP@3
# final_map3 = np.mean(scores)

# print("TF-IDF Pipeline MAP@3:", round(final_map3, 4))

ANS:TF-IDF Pipeline MAP@3: 0.2962